In [37]:

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [38]:
# Load Data and Seperate Target
df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs","Property_ID"])

y = df["Price_INR_Lakhs"]

In [39]:
# Train Val Split
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=42)

In [40]:
# Feature Listing 
categorical_features= [ "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]


In [41]:
# Creating Pipline for  Preprocessing
numerical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy = "most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])


In [42]:
# # Apply Pipeline

preprocessor = ColumnTransformer([
    ("num",numerical_pipeline,numerical_features),
    ("cat",categorical_pipeline,categorical_features)
])

In [43]:
# Create Model

model = Pipeline([
    ("preprocess",preprocessor),
    ("regressor",RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

In [44]:
# Train Model and predict

model.fit(X_train,y_train)

y_pred = model.predict(X_val)

In [45]:
print(y_pred)

[404.4639391   85.79189542  43.85305257 ...  60.86079635  72.03218576
  38.46901532]


In [46]:
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_val, y_pred)

print("Random Forest Baseline Results")
print("-" * 40)
print(f"Mean Absolute Error : {mae:.2f} lakhs")
print(f"Root Mean Squared Error : {rmse:.2f} lakhs")
print(f"R2 Score : {r2:.4f}")

Random Forest Baseline Results
----------------------------------------
Mean Absolute Error : 34.05 lakhs
Root Mean Squared Error : 70.43 lakhs
R2 Score : 0.7760


In [48]:
# Compare Actual vs predicted

print("\nActual vs Predicted (first 10 rows)")
comparison = pd.DataFrame({
    "Actual Price":y_val.iloc[:10].values,
    "Predicted Price":y_pred[:10]
})

print(comparison.round(2).to_string(index=False))


Actual vs Predicted (first 10 rows)
 Actual Price  Predicted Price
       333.73           404.46
       112.09            85.79
        66.90            43.85
        30.08            50.33
       177.49           157.05
       120.97           119.82
       348.51           306.71
       152.95           112.47
       121.86           158.70
       173.84           141.13
